# AstroShield — optimización evolutiva de una constelación

**Entorno recomendado:** Google Colab. Ejecuta las celdas en orden desde `Entorno de ejecución > Reiniciar sesión y ejecutar todo`.

**Modelo educativo:** 8 satélites, 6 regiones, radio de observación nominal de 300 km, 3 zonas restringidas y 2 objetos ocupados. La cobertura se aproxima usando el punto subsatelital y el algoritmo verifica restricciones en muestras temporales discretas; estos supuestos no representan una simulación orbital de alta fidelidad.

Si Plotly no se muestra o no se dispone de conexión, el algoritmo y sus métricas pueden ejecutarse de todas formas; las líneas continentales detalladas son un recurso opcional de la visualización.


In [ ]:
# ============ CELDA 1: imports, constantes y configuración ============
from __future__ import annotations
import json, time
from dataclasses import dataclass, field, asdict
from typing import Callable, Sequence
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

MU_EARTH = 398600.4418    # km^3/s^2
R_EARTH  = 6371.0         # km (Tierra esférica)
OMEGA_E  = 7.2921159e-5   # rad/s, rotación terrestre

@dataclass(frozen=True)
class Region:
    name: str
    lat: float
    lon: float
    priority: str   # "alta" | "media" | "normal"

DEFAULT_REGIONS = (
    Region("Península de Yucatán",  20.5,  -89.0, "alta"),
    Region("Amazonía brasileña",    -3.5,  -62.0, "alta"),
    Region("California",            36.8, -119.4, "media"),
    Region("Sahel",                 15.0,    0.0, "media"),
    Region("Japón",                 36.0,  138.0, "media"),
    Region("Interior de Australia", -25.0, 133.0, "normal"),
)
PRIORITY_WEIGHTS = {"alta": 3.0, "media": 2.0, "normal": 1.0}

@dataclass
class Config:
    # --- problema ---
    n_sats: int = 8
    radius_km: float = 300.0
    regions: tuple = DEFAULT_REGIONS
    priority_weights: dict = field(default_factory=lambda: dict(PRIORITY_WEIGHTS))
    alt_range_km: tuple = (500.0, 1200.0)
    inc_range_deg: tuple = (0.0, 98.0)
    sim_hours: float = 24.0
    dt_s: float = 30.0
    # --- algoritmo evolutivo ---
    pop_size: int = 60
    generations: int = 100
    seed: int = 42
    tournament_k: int = 3
    crossover_prob: float = 0.9
    blx_alpha: float = 0.3
    mutation_prob_gene: float = 0.10
    sigma_start: float = 0.15
    sigma_end: float = 0.02
    reinit_prob: float = 0.02
    elitism: int = 2
    # --- terminación ---
    patience: int = 30
    tol: float = 1e-4
    target_fitness: float | None = None
    max_time_s: float | None = None

    def time_grid(self):
        return np.arange(0.0, self.sim_hours * 3600.0, self.dt_s)

    def weights_array(self):
        return np.array([self.priority_weights[r.priority] for r in self.regions], float)

    def region_units(self):
        """Vectores unitarios ECEF de las regiones, shape (J, 3)."""
        lat = np.radians([r.lat for r in self.regions])
        lon = np.radians([r.lon for r in self.regions])
        return np.stack([np.cos(lat)*np.cos(lon), np.cos(lat)*np.sin(lon), np.sin(lat)], -1)

In [ ]:
# ============ CELDA 2: propagación, cobertura y evaluación ============
N_GENES = 4                                   # [altitud, inclinación, RAAN, fase]
CIRC = np.array([False, False, True, True])   # genes circulares

def decode_params(genome, cfg: Config) -> dict:
    """Genoma normalizado -> elementos orbitales legibles (grados / km)."""
    g = np.asarray(genome, float).reshape(cfg.n_sats, N_GENES)
    a0, a1 = cfg.alt_range_km
    i0, i1 = cfg.inc_range_deg
    alt = a0 + g[:, 0] * (a1 - a0)
    return {"altitude_km": alt,
            "inclination_deg": i0 + g[:, 1] * (i1 - i0),
            "raan_deg": 360.0 * g[:, 2],
            "phase_deg": 360.0 * g[:, 3],
            "period_min": 2*np.pi*np.sqrt((R_EARTH+alt)**3/MU_EARTH)/60.0}

def encode_orbits(orbits: Sequence[dict], cfg: Config) -> np.ndarray:
    """Inverso de decode_params: lista de dicts -> genoma (útil para validadores externos)."""
    a0, a1 = cfg.alt_range_km; i0, i1 = cfg.inc_range_deg
    g = np.zeros((len(orbits), N_GENES))
    for k, o in enumerate(orbits):
        g[k] = [(o["altitude_km"]-a0)/(a1-a0), (o["inclination_deg"]-i0)/(i1-i0),
                (o["raan_deg"] % 360)/360.0, (o["phase_deg"] % 360)/360.0]
    return np.clip(g, 0, 1)

def _unit_vectors(genome, cfg, t, frame="ecef"):
    """Vectores unitarios de posición (S,T,3) y radio orbital a (S,)."""
    p = decode_params(genome, cfg)
    a = R_EARTH + p["altitude_km"]
    inc, raan, nu0 = np.radians(p["inclination_deg"]), np.radians(p["raan_deg"]), np.radians(p["phase_deg"])
    n = np.sqrt(MU_EARTH / a**3)                              # rad/s
    u = nu0[:, None] + n[:, None] * t[None, :]                # argumento de latitud (S,T)
    cu, su = np.cos(u), np.sin(u)
    ci, si = np.cos(inc)[:, None], np.sin(inc)[:, None]
    cO, sO = np.cos(raan)[:, None], np.sin(raan)[:, None]
    x = cO*cu - sO*ci*su
    y = sO*cu + cO*ci*su
    z = si*su
    if frame == "ecef":                                       # rotar con la Tierra
        th = OMEGA_E * t
        c, s = np.cos(th)[None, :], np.sin(th)[None, :]
        x, y = x*c + y*s, -x*s + y*c
    return np.stack([x, y, z], -1), a

def propagate(genome, cfg: Config, t=None, frame="ecef"):
    """Posiciones en km (S,T,3), y lat/lon del punto subsatelital. Para visualización 3D."""
    t = cfg.time_grid() if t is None else np.asarray(t, float)
    unit, a = _unit_vectors(genome, cfg, t, frame)
    lat = np.degrees(np.arcsin(unit[..., 2]))
    lon = np.degrees(np.arctan2(unit[..., 1], unit[..., 0]))
    return {"t_s": t, "pos_km": unit * a[:, None, None], "lat_deg": lat, "lon_deg": lon}

def _coverage(unit, region_units, radius_km):
    """Booleano (T,J): ¿alguna huella de satélite cubre la región en ese instante?"""
    cos_thr = np.cos(radius_km / R_EARTH)
    dots = np.einsum("stk,jk->stj", unit, region_units)
    return (dots >= cos_thr).any(axis=0)

def _longest_false_run(cov):
    out = np.zeros(cov.shape[1])
    for j in range(cov.shape[1]):
        x = ~cov[:, j]
        if x.any():
            d = np.diff(np.concatenate(([0], x.astype(int), [0])))
            out[j] = (np.where(d == -1)[0] - np.where(d == 1)[0]).max()
    return out

def _fitness(genome, cfg, t, region_units, w):
    unit, _ = _unit_vectors(genome, cfg, t)
    frac = _coverage(unit, region_units, cfg.radius_km).mean(axis=0)
    return float((w * frac).sum() / w.sum())

def fitness_value(genome, cfg: Config) -> float:
    """API pública: cobertura ponderada [0,1] de un genoma (sin penalizaciones)."""
    return _fitness(genome, cfg, cfg.time_grid(), cfg.region_units(), cfg.weights_array())

def describe_solution(genome, cfg: Config) -> dict:
    """Evaluación detallada e independiente (punto de entrada para validadores)."""
    unit, _ = _unit_vectors(genome, cfg, cfg.time_grid())
    cov = _coverage(unit, cfg.region_units(), cfg.radius_km)
    frac, w = cov.mean(axis=0), cfg.weights_array()
    gaps = _longest_false_run(cov) * cfg.dt_s / 3600.0
    p = decode_params(genome, cfg)
    return {
        "weighted_coverage": float((w * frac).sum() / w.sum()),
        "mean_coverage": float(frac.mean()),
        "regions": [{"name": r.name, "lat": r.lat, "lon": r.lon, "priority": r.priority,
                     "weight": float(w[j]), "coverage_pct": float(100*frac[j]),
                     "max_gap_h": float(gaps[j])} for j, r in enumerate(cfg.regions)],
        "satellites": [{k: float(v[s]) for k, v in p.items()} for s in range(cfg.n_sats)],
    }

In [ ]:
# ============ CELDA 3: operadores evolutivos y ejecución ============
Validator = Callable[[np.ndarray, Config], float]   # devuelve penalización >= 0

def _repair(g):
    return np.where(CIRC, g % 1.0, np.clip(g, 0.0, 1.0))

def _tournament(fit, rng, k):
    idx = rng.integers(0, len(fit), size=k)
    return idx[np.argmax(fit[idx])]

def _blx(a, b, rng, alpha):
    """BLX-α; en genes circulares usa el arco corto entre a y b."""
    d = b - a
    d = np.where(CIRC, (d + 0.5) % 1.0 - 0.5, d)
    lo, hi = np.minimum(0, d), np.maximum(0, d)
    w = hi - lo
    return a + rng.uniform(lo - alpha*w, hi + alpha*w)

def _crossover(p1, p2, rng, cfg):
    c1, c2 = p1.copy(), p2.copy()
    if rng.random() > cfg.crossover_prob:
        return c1, c2
    for s in range(cfg.n_sats):
        if rng.random() < 0.5:                       # intercambio de satélite completo
            if rng.random() < 0.5:
                c1[s], c2[s] = p2[s].copy(), p1[s].copy()
        else:                                        # mezcla BLX
            c1[s], c2[s] = _blx(p1[s], p2[s], rng, cfg.blx_alpha), _blx(p2[s], p1[s], rng, cfg.blx_alpha)
    return _repair(c1), _repair(c2)

def _mutate(g, rng, cfg, sigma):
    g = g + (rng.random(g.shape) < cfg.mutation_prob_gene) * rng.normal(0, sigma, g.shape)
    re = rng.random(cfg.n_sats) < cfg.reinit_prob    # reinicio de satélites completos
    if re.any():
        g[re] = rng.random((int(re.sum()), N_GENES))
    return _repair(g)

@dataclass
class EvolutionResult:
    best_genome: np.ndarray
    best_fitness: float          # lo optimizado (incluye penalizaciones de validadores)
    metrics: dict                # evaluación detallada sin penalizaciones
    history: pd.DataFrame
    config: Config
    n_evals: int
    runtime_s: float
    termination_reason: str

    def to_dict(self) -> dict:
        return {"best_genome": self.best_genome.tolist(), "best_fitness": self.best_fitness,
                "metrics": self.metrics, "history": self.history.to_dict("records"),
                "config": asdict(self.config), "n_evals": self.n_evals,
                "runtime_s": self.runtime_s, "termination_reason": self.termination_reason}

    def save_json(self, path="astroshield_result.json"):
        with open(path, "w", encoding="utf-8") as f:
            json.dump(self.to_dict(), f, ensure_ascii=False, indent=2)
        return path

def run_evolution(cfg: Config | None = None, validators: Sequence[Validator] = (),
                  callback: Callable | None = None, verbose: bool = True) -> EvolutionResult:
    cfg = cfg or Config()
    assert cfg.pop_size >= 2 and 0 <= cfg.elitism < cfg.pop_size and cfg.generations >= 1
    rng = np.random.default_rng(cfg.seed)
    t, ru, w = cfg.time_grid(), cfg.region_units(), cfg.weights_array()

    def score(g):
        f = _fitness(g, cfg, t, ru, w)
        for v in validators:
            f -= float(v(g, cfg))
        return f

    t0 = time.perf_counter()
    pop = rng.random((cfg.pop_size, cfg.n_sats, N_GENES))
    fit = np.array([score(g) for g in pop]); n_evals = len(pop)
    best_g, best_f = pop[np.argmax(fit)].copy(), float(fit.max())
    history, stall, reason = [], 0, "max_generations"

    def log(gen, sigma):
        row = {"gen": gen, "best": float(fit.max()), "best_so_far": best_f, "mean": float(fit.mean()),
               "worst": float(fit.min()), "std": float(fit.std()),
               "diversity": float(pop.reshape(len(pop), -1).std(axis=0).mean()),
               "sigma": sigma, "n_evals": n_evals, "elapsed_s": time.perf_counter() - t0}
        history.append(row); return row

    log(0, cfg.sigma_start)
    for gen in range(1, cfg.generations + 1):
        sigma = cfg.sigma_start + (cfg.sigma_end - cfg.sigma_start) * (gen - 1) / max(1, cfg.generations - 1)
        order = np.argsort(fit)[::-1]
        elite, elite_fit = pop[order[:cfg.elitism]].copy(), fit[order[:cfg.elitism]].copy()
        n_child, kids = cfg.pop_size - cfg.elitism, []
        while len(kids) < n_child:
            p1 = pop[_tournament(fit, rng, cfg.tournament_k)]
            p2 = pop[_tournament(fit, rng, cfg.tournament_k)]
            c1, c2 = _crossover(p1, p2, rng, cfg)
            kids += [_mutate(c1, rng, cfg, sigma), _mutate(c2, rng, cfg, sigma)]
        kids = np.array(kids[:n_child])
        kid_fit = np.array([score(g) for g in kids]); n_evals += len(kids)
        pop, fit = np.concatenate([elite, kids]), np.concatenate([elite_fit, kid_fit])

        if fit.max() > best_f:
            improved = fit.max() - best_f > cfg.tol
            best_f, best_g = float(fit.max()), pop[np.argmax(fit)].copy()
            stall = 0 if improved else stall + 1
        else:
            stall += 1
        row = log(gen, sigma)
        if verbose and (gen % 10 == 0 or gen == 1):
            print(f"gen {gen:4d} | mejor {best_f:.4f} | media {row['mean']:.4f} | div {row['diversity']:.3f}")

        if callback and callback(gen, row, best_g):                       reason = "callback"; break
        if cfg.target_fitness is not None and best_f >= cfg.target_fitness: reason = "target_fitness"; break
        if stall >= cfg.patience:                                          reason = "stagnation"; break
        if cfg.max_time_s and time.perf_counter() - t0 > cfg.max_time_s:   reason = "max_time"; break

    return EvolutionResult(best_g, best_f, describe_solution(best_g, cfg), pd.DataFrame(history),
                           cfg, n_evals, time.perf_counter() - t0, reason)

In [ ]:
# ============ CELDA 3.5: validadores de restricciones espaciales ============
def angular_distance_deg(lat1, lon1, lat2, lon2):
    """Calcula la distancia angular en grados entre dos conjuntos de coordenadas sobre una esfera."""
    r_lat1, r_lon1 = np.radians(lat1), np.radians(lon1)
    r_lat2, r_lon2 = np.radians(lat2), np.radians(lon2)
    cos_d = np.sin(r_lat1) * np.sin(r_lat2) + np.cos(r_lat1) * np.cos(r_lat2) * np.cos(r_lon1 - r_lon2)
    return np.degrees(np.arccos(np.clip(cos_d, -1.0, 1.0)))

def make_spatial_validator(center_lat: float, center_lon: float, exclusion_radius_deg: float, penalty_weight: float = 1.0) -> Validator:
    """Crea un validador que penaliza si algún satélite entra en la zona de exclusión durante la simulación."""
    def validator(genome: np.ndarray, cfg: Config) -> float:
        tr = propagate(genome, cfg, frame="ecef")
        lat_sats = tr["lat_deg"]  # (S, T)
        lon_sats = tr["lon_deg"]  # (S, T)

        # Distancia angular de cada satélite en cada instante al centro restringido
        dist = angular_distance_deg(lat_sats, lon_sats, center_lat, center_lon)

        # Si la distancia es menor que el radio de exclusión, hay invasión
        invasions = exclusion_radius_deg - dist
        max_invasions_per_sat = np.maximum(0.0, invasions).max(axis=1)

        # Penalización total ponderada
        return float(np.sum(max_invasions_per_sat) * penalty_weight)
    return validator

# Definición de validadores oficiales según el enunciado
val_atlantico_norte = make_spatial_validator(45.0, -35.0, 12.0, penalty_weight=2.0)
val_pacifico_sur = make_spatial_validator(-30.0, -120.0, 10.0, penalty_weight=2.0)
val_polar_norte = make_spatial_validator(78.0, 30.0, 8.0, penalty_weight=2.0)
val_objeto_a = make_spatial_validator(10.0, 45.0, 5.0, penalty_weight=2.0)
val_objeto_b = make_spatial_validator(-15.0, 110.0, 5.0, penalty_weight=2.0)

ALL_VALIDATORS = [
    val_atlantico_norte,
    val_pacifico_sur,
    val_polar_norte,
    val_objeto_a,
    val_objeto_b
]

def combined_spatial_validator(genome: np.ndarray, cfg: Config) -> float:
    """Suma las penalizaciones espaciales usando una sola propagación por candidato."""
    tr = propagate(genome, cfg, frame="ecef")
    lat_sats, lon_sats = tr["lat_deg"], tr["lon_deg"]
    zones = [
        (45.0, -35.0, 12.0, 2.0),
        (-30.0, -120.0, 10.0, 2.0),
        (78.0, 30.0, 8.0, 2.0),
        (10.0, 45.0, 5.0, 2.0),
        (-15.0, 110.0, 5.0, 2.0),
    ]
    total = 0.0
    for lat_c, lon_c, radius, weight in zones:
        dist = angular_distance_deg(lat_sats, lon_sats, lat_c, lon_c)
        total += float(np.maximum(0.0, radius - dist).max(axis=1).sum()) * weight
    return total
